# ADNI

## INIT

In [1]:
from data_model.DataCleaner import DataCleaner, update_variables_support_file
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

dataCleaner = DataCleaner(support_file_path='ADNI_variables_statistics.xlsx')
client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
file_codes = ['ADAS', 'FAQ', 'CDR', 'MOCA']

In [3]:
search = client.query_files(
    query={'custom.level' : 'raw', 'custom.source' : 'ADNI', 'custom.file_code' : file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())
print(len(zip_files.keys()))


dict_keys(['ADAS_28Oct2025.csv', 'CDR_28Oct2025.csv', 'FAQ_28Oct2025.csv', 'MOCA_28Oct2025.csv'])
4


# Support file managment
operazione per popolare il file support file per i file considerati

In [4]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

In [5]:
for file_name in list(zip_files.keys()):
    df = zip_files[file_name]
    infoSupportFile = InfoSupportFile(support_file, df, file_name)
    # delate the rows of the support file that are not in the df
    support_file, file_code = infoSupportFile.filter_variables()
    print(file_code)
    if file_code not in list(support_file['file_code']):
        print('not found in excel')
        continue
    # find the population variable code, and if not in support_file, add it
    pop, support_file = infoSupportFile.find_population_variable()
    # get the variable info and add it to the support_file    
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            infoSupportFile.get_varible_info(key)
# save the updated support file
save_df(df_to_save=support_file, output_path=support_file_path)

ADAS
CDR
FAQ
MOCA


In [6]:
new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)

The ADNI_variables_cleaned1 file has been updated with the new file_code: ['FAQ', 'ADAS', 'CDR', 'MOCA']
Open the file and verify it, if needed update the variables names and metadata


## IF SUPPORT FILE already populated

In [ ]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)


In [3]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'

In [ ]:
new_support_file_name = 'ADNI_variables_cleaned1'

Open the new_support_file and fill in the new variable codes.

# FILE SPECIFIC DATA CLEANING 1


## APOE

### APOERES

In [ ]:
file_code = 'APOERES'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['GENOTYPE'])


In [ ]:
print('prima', len(df_new),'\ndopo ', len(no_none_df))

In [ ]:
no_none_df.head()

In [ ]:
standardised_df = dataCleaner.uniform_APOE_format(df=no_none_df, col_name='GENOTYPE')
processed_df = dataCleaner.APOE_4_count(df=standardised_df, col_name='GENOTYPE')

In [ ]:
display(processed_df.head())

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['APOE_4'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
final_df

In [ ]:
# optaining automatically info to save the file
lst_population =  infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### YASSIN_CSF & YASSIN_PLASMA

In [ ]:
file_code = 'YASSINE_PLASMA'       #CSF
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['Phenotype'])


In [ ]:
print('prima', len(df_new),'\ndopo ', len(no_none_df))

In [ ]:
no_none_df.head()

In [ ]:
no_none_df.keys()

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= ['Phenotype'])

In [ ]:
standardised_df = dataCleaner.uniform_APOE_format(df=datefix_df, col_name='Phenotype')
processed_df = dataCleaner.APOE_4_count(df=standardised_df, col_name='Phenotype')

In [ ]:
display(processed_df.head())

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['APOE_4'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
final_df

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:

lst_population =  ['ADNI1', 'ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
new_file_name

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Abeta & Tau in CSF - Elecsys

### UPENNBIOMK_ROCHE_ELECSYS

In [ ]:
file_code = 'UPENNBIOMK_ROCHE_ELECSYS'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'TAU', 'PTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df)



In [ ]:
final_df.head()

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENNBIOMK_ADNIDIAN_ES_2017

In [ ]:
file_code = 'UPENNBIOMK_ADNIDIAN_ES_2017'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA','TAU','PTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df)


In [ ]:
final_df

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Abeta Tau altri metodi

### EUROIMMUN

In [ ]:
file_code = 'EUROIMMUN'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['BETA_AMYLOID_1_40', 'BETA_AMYLOID_1_42']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [ ]:
final_df

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:

final_df = final_df[final_df['EXAMDATE'].notna()]
final_df

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### FUJIREBIOABETA

In [ ]:
file_code = 'FUJIREBIOABETA'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'ABETA40']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['EXAMDATE'])

In [ ]:
no_none_df

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### SALADAX_BIOMEDICAL

In [ ]:
file_code = 'SALADAX_BIOMEDICAL'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'TOTALTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['EXAMDATE'])

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### MESOSCALE

In [ ]:
file_code = 'MESOSCALE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA40', 'ABETA42', 'TAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['ABETA42'])

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['DRAWDTE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDTE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = 'ADNI_MESOSCALE_23Oct2025_01.csv'

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENNBIOMK_MASTER
tenere solo parametri ribilanciati e non quelli raw
filtrare per tenere per ciascuna visita solo la riga con BATCH == MEDIAN altrimenti calcolare np.median tra i valori, se una sola visita ==> prendere quel valore e basta

In [ ]:
file_code = 'UPENNBIOMK_MASTER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA', 'PTAU', 'TAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['ABETA'])

In [ ]:
reduced_df = dataCleaner.get_mean_row_per_visit(no_none_df)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(reduced_df, ['DRAWDTE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDTE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENN_2DUPLC_CRM

In [ ]:
file_code = 'UPENN_2DUPLC_CRM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA40', 'ABETA42']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['ABETA42'])

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
final_df

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Volumi

### UCSDVOL

In [ ]:
file_code = 'UCSDVOL'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['BRAIN', 'EICV', 'VENTRICLES', 'LHIPPOC', 'RHIPPOC']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# Convert QCPASS values from 1/0 to 'complete'/'partial'
no_none_df = dataCleaner.convert_qcpass_values(no_none_df, col_name='QCPASS')
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='QCPASS')

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSF Longitudinal dataset

In [ ]:
file_codes = ['UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL']
file_code = file_codes[3]

In [ ]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
if file_code == 'UCSFFSL': #the other UCSF longitudinal files have just partial immages segmentation
    no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
if file_code == 'UCSFFSL':
    lst_population = ['ADNI1','ADNIGO','ADNI2']
else:
    lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX 
complete    4485\
partial        1\
hanno solo VISITCODE e non VISITCODE2 inoltre non hanno info sulla popolazione --> da inserire manualmente?

In [ ]:
file_code = 'UCSFFSX' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
######### ERRORE DA RISOLVERE
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)



infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFX7

partial     11091\
complete      849

In [ ]:
file_code = 'UCSFFSX7' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') 

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
# update the new info support file
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX6
complete    2222\
partial       18

In [ ]:
file_code = 'UCSFFSX6' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)



infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

In [ ]:
len(final_df['RID'].value_counts()[final_df['RID'].value_counts() == 1])

### UCSFFSX51

In [ ]:
file_code = 'UCSFFSX51' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has no status column

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX51_ADNI1_3T
partial    484


In [ ]:
file_code = 'UCSFFSX51_ADNI1_3T' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has only partial segmentations in the STATUS ==> run funzione ma solo per verifica no modifica dataset... resterebbe vuoto? 
# VERIFICARE ma non da usare
test_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENN_ROI_MARS

In [ ]:
file_code = 'UPENN_ROI_MARS' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['R702', 'R525', 'R517', 'R122', 'R123', 'R116', 'R117', 'R47', 'R48']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['R702']) # se manca ICV non si può normalizzare
# this file has only partial segmentations in the STATUS ==> run funzione ma solo per verifica no modifica dataset... resterebbe vuoto? 
# VERIFICARE ma non da usare
test_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(test_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Multiparametre dataset


### ADNI MERGE

In [ ]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 


In [ ]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['AGE_bl', 'VISIT_MONTH'], prefix='raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE


In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### PTDEMOG

In [ ]:
file_code = 'PTDEMOG'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOB'], birth_year=row['PTDOBYY']), axis=1)

In [ ]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
final_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['AGE', 'VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### ADSP_PHC_BIOMARKER

In [ ]:
file_code = 'ADSP_PHC_BIOMARKER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADSP_PHC_BIOMARKER'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PHC_Tau', 'PHC_pTau', 'PHC_AB42', 'Tau_RAW', 'pTau_RAW', 'AB42_RAW', 'AT_class']
single_column_required = ['PHC_Diagnosis']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['DRAWDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = dataCleaner.binarization_gender(datefix_df, col_name='PHC_Sex')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PHC_Education')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PHC_Ethnicity')
processed_df, new_var = dataCleaner.convert_to_dummies_ATNC_profile(processed_df, col_name='AT_class')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='PHC_Diagnosis')

In [ ]:
new_var

In [ ]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
processed_df['PHC_Race'] = processed_df['PHC_Race'].map(mapping)

In [ ]:
filtered_df =dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'] + new_var, remove_var=['VISCODE'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)

In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### BLCHANGE

In [ ]:
file_code = 'BLCHANGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'BLCHANGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['BCMMSE', 'BCADAS', 'BCPREDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='BCPREDX')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### DXSUM

In [ ]:
file_code = 'DXSUM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
        }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['DXNORM', 'DXMCI', 'DXNODEP']
required_column = ['DIAGNOSIS']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, required_column)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='DIAGNOSIS')
final_df = dataCleaner.to_date_format(final_df, ['EXAMDATE'])

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### ADNI_DIAN_COMPARISON

In [ ]:
file_code = 'ADNI_DIAN_COMPARISON'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 


In [ ]:
# Important columns
# in questo caso non metto altri filtri perche mancano davvero tante info e preferisco tenere un file molto frammentato, altrimenti si potrebbero togliere le righe che non hanno queste info, ma perderemmo informazioni su pop che magari sono riutilizzabili
single_column_required = ['EXAMDATE']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
#no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, single_column_required)

In [ ]:
columns_must_be_verified = ['MR_TOTV_INTRACRANIAL', 'MR_TOTV_HIPPOCAMPUS', 'CSF_ELC_AB42', 'CSF_ELC_PTAU', 'CSF_ELC_TAU', 'CSF_ELC_AB40', 'CSF_ELC_AB4240', 'MSP_AB40', 'MSP_AB42']
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)

In [ ]:
# CATHEGORIZATION Gender
col_name = 'GENDER'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name]
# mapp gender in to classes: 0 = 'female' --> 0, 1 = 'male' --> 1 --> non necessario perchè già nel formato corretto

In [ ]:
# CATEGORIZZAZIONE Status Maritale
col_name = 'MARISTAT'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name]
# Map marital status to classes: 1 ='married' -> 1, 6 = 'Living as married' --> 1, 3 = 'divorced' -> 2, 4 = 'separated' --> 2,  2 ='widowed' -> 3, 5 = 'never married' -> 0, 7 = 'Other' --> nan, 9 = 'Unknown' --> nan
mapping = {1: 1, 6: 1, 3: 2, 4: 2, 2: 3, 5: 0, 1.0: 1, 6.0: 1, 3.0: 2, 4.0: 2, 2.0: 3, 5.0: 0}
processed_df[col_name] = processed_df[col_name].map(mapping)

In [ ]:
# CATEGORIZZAZIONE Etnicity
col_name = 'HISPANIC'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name]
# mapp Ethnicity in to classes: 0 = 'not hisp/latino' --> 1, 1 = 'hisp/latino' --> 0
processed_df[col_name] = processed_df[col_name].map(lambda x: 1 if x in [0, 0.0] else (0 if x in [1, 1.0] else np.nan))

In [ ]:
# CATEGORIZZAZIONE Race
col_name = 'RACE'
# mi assicuro che tutta la colonna sia di interi
processed_df[col_name] = processed_df[col_name]
# Map race to classes: 1 = 'White' --> 5,  2 = 'Black' --> 4, 5 = 'Asian'--> 2, 3 = 'Am Indian/Alaskan' --> 1, 4 = 'Hawaiian/Other PI' -->3)
mapping = {1: 5, 2: 4, 3: 1, 4: 3, 5: 2, 1.0: 5, 2.0: 4, 3.0: 1, 4.0: 3, 5.0: 2}
processed_df[col_name] = processed_df[col_name].map(mapping)

In [ ]:
processed_df = dataCleaner.uniform_APOE_format(df=no_none_df, col_name='DIAN_APOE')
processed_df = dataCleaner.APOE_4_count(df=processed_df, col_name='DIAN_APOE')

In [ ]:
filtered_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
AbT_df, ratios_var2 = dataCleaner.get_abeta_tau_ratios(AbT_df, AB42='AB42_ms', AB40='AB40_ms')

final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df)

In [ ]:
final_df

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
   
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Clinical scales

### MMSE

In [ ]:
file_code = 'MMSE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'MMSE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['MMSCORE']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### ADAS 13 & 11

In [7]:
file_code = 'ADAS'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [8]:
# Important columns
columns_must_be_verified = ['TOTSCORE', 'TOTAL13']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [9]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  2
Adopted visit selection strategy:
 VISITCODE priority    2
Name: count, dtype: int64


In [10]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [11]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [12]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [14]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### FAQ

In [4]:
file_code = 'FAQ'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [5]:
# Important columns
columns_must_be_verified = ['FAQTOTAL']

no_unknow_df = dataCleaner.replace_unknown_values(df_new, new_nans=[-1])
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [6]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  6
Adopted visit selection strategy:
 Equal values          5
VISITCODE priority    1
Name: count, dtype: int64


In [7]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [8]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [9]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [10]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### CDR

In [11]:
file_code = 'CDR'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [12]:
# Important columns
columns_must_be_verified = ['CDGLOBAL', 'CDRSB']

no_unknow_df = dataCleaner.replace_unknown_values(df_new, new_nans=[-1])
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [17]:
print(len(df_new))
print(len(no_none_df))

14598
14354


In [13]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  4
Adopted visit selection strategy:
 Equal values          3
VISITCODE priority    1
Name: count, dtype: int64


In [14]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], prefix='raw')  

In [18]:
print(len(df_new))
print(len(final_df))

14598
14350


In [19]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [20]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [21]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### MOCA

In [23]:
file_code = 'MOCA'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [24]:
# Important columns
columns_must_be_verified = ['MOCA']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, ['VISCODE2', 'VISDATE'])
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)

In [25]:
print(len(df_new))
print(len(no_none_df))

8944
3689


In [26]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [27]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')  

In [29]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

In [30]:
print(len(df_new))
print(len(final_df))

8944
3689


Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [31]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [32]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

In [ ]:
### KEEEP --> tentativo di calcolare MOCA dagli altri valori ma qualcosa non torna
'''
df_focus['VIS_SPACE'] = df_focus['TRAILS']+df_focus['CUBE']+df_focus['CLOCKCON']+df_focus['CLOCKNO']+df_focus['CLOCKHAN']
df_focus['NAMING']= df_focus['LION']+df_focus['RHINO']+df_focus['CAMEL']
df_focus['RECALL'] = df_focus['DELW1']+df_focus['DELW2']+df_focus['DELW3']+df_focus['DELW4']+df_focus['DELW5']

serial_sum = df_focus['SERIAL1']+df_focus['SERIAL2']+df_focus['SERIAL3']+df_focus['SERIAL4']+df_focus['SERIAL5']
serial_sum = serial_sum.map(lambda x: 0 if x <= 1 else 1 if x in [2,3] else 2 if x == 4 else 3 if x == 5 else print('serial_sum:', x, 'out of range'))
letter_score = df_focus['LETTERS'].map(lambda x: 1 if x <= 1 else 0 if x > 1 else print('letter_score:', x, 'out of range'))
df_focus['ATTENTION'] = serial_sum + letter_score + df_focus['DIGFOR'] + df_focus['DIGBACK']

fluency_score = df_focus['FFLUENCY'].map(lambda x: 1 if x>=11 else 0 if x<11 else print('fluency_score:', x, 'out of range'))
df_focus['LANGUAGE'] = fluency_score + df_focus['REPEAT1'] + df_focus['REPEAT2']

df_focus['ABSTRACTION'] = df_focus['ABSTRAN'] + df_focus['ABSMEAS']
df_focus['ORIENTATION'] = df_focus['DATE'] + df_focus['MONTH'] + df_focus['YEAR'] + df_focus['DAY']+df_focus['PLACE']+df_focus['CITY']

df_focus['MOCA_tot'] = df_focus['VIS_SPACE'] + df_focus['NAMING'] + df_focus['RECALL'] + df_focus['ATTENTION'] + df_focus['LANGUAGE']+ df_focus['ABSTRACTION'] + df_focus['ORIENTATION']
'''
